### Building Chatbot With Multiple Tools using LangGraph

Bulid a chatbot with tools from arxiv, wikipedia search and multiple functions

In [17]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
import arxiv

import os
from dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_groq import ChatGroq
from pprint import pprint
from langchain_core.messages import HumanMessage, AIMessage

# Chatbot with LangGraph
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# State schema
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages

In [18]:
# Create a client with a 
client = arxiv.Client(
    page_size=10,
    delay_seconds=5,
    num_retries=3,
)

api_wrapper_arxiv = ArxivAPIWrapper(
    top_k_results=2, # means 
    doc_content_chars_max=2000, # 
    load_max_docs=2 # 
)

arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [ ]:
# Use arXiv search keywords instead of a full natural-language question
from time import sleep

query = 'multimodal latest research'
for attempt in range(4):
	try:
		result = arxiv.invoke(query)
		print(result)
		break
	except Exception as e:
		print(f'Attempt {attempt + 1} failed: {e}')
		if attempt == 3:
			raise
		sleep(10)

HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=multimodal+latest+research&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)

In [ ]:
api_wrapper_wikipdia = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wikipdia)
print(wiki.name)

wiki.invoke('What is the difference between machine learning and deep learning?')

wikipedia


'Page: Machine learning\nSummary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions. Within a subdiscipline of machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.\nStatistics and mathematical optimisation (mathematical programming) methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.\nFrom a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empiric

In [ ]:
load_dotenv()

os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [ ]:
# Tavily search tools

tavily = TavilySearchResults() 
tavily

TavilySearchResults(api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********')))

In [ ]:
tavily.invoke('Get me the ML, DL, LLM and Agentic AI latest news')

[{'title': '"Understanding the Layers of AI: ML, DL, LLM, GenAI, Agentic AI" | Shubhankit Sirvaiya posted on the topic | LinkedIn',
  'url': 'https://www.linkedin.com/posts/shubhankitsirvaiya_confused-between-ml-dl-llm-genai-and-activity-7376820253990899712-HqTd',
  'content': '🤖 Move into LLMs & GenAI →\n🧑🤝🧑 End with Agentic AI + enterprise-grade engineering.\nSo you don’t just learn buzzwords — you learn the entire stack of AI, end-to-end.\n👉 Sign up here: [...] Built on top of LLMs (and other generative models).\nFocused on creation: text, code, images, music.\nExamples: ChatGPT, MidJourney, Copilot.\n🧑🤝🧑 Agentic AI\nThe next layer.\nNot just generating, but acting: reasoning, planning, using tools, collaborating with other agents.\nExample: a multi-agent system that books travel, processes expenses, and updates reports autonomously.\n💡 How to Learn These Layers\nYou can’t skip.\nStart with ML → get your fundamentals right.\nMove into DL → understand neural nets.\nLearn how LLMs are

In [ ]:
#  Merge all the tools
tools = [arxiv, wiki, tavily]

# Initializing LLM model
llm = ChatGroq(model='llama-3.1-8b-instant')

llm_with_tools = llm.bind_tools(tools)

llm_with_tools

RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001F63DAE9A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F63DB28CD0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'arxiv', 'description': 'A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'search query to look up', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, 

In [ ]:
llm_with_tools.invoke([
    HumanMessage(content='What is the waether in FCT Abuja?')
])

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tvg8ktpx4', 'function': {'arguments': '{"query":"FCT Abuja weather today"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 558, 'total_tokens': 581, 'completion_time': 0.038649021, 'prompt_time': 0.071555154, 'queue_time': 0.010724857, 'total_time': 0.110204175}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019d3578-e21e-7f92-b865-514efe6bb1d3-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'FCT Abuja weather today'}, 'id': 'tvg8ktpx4', 'type': 'tool_call'}], usage_metadata={'input_tokens': 558, 'output_tokens': 23, 'total_tokens': 581})

In [ ]:
llm_with_tools.invoke([
    HumanMessage(content='What is the waether in FCT Abuja?')
]).tool_calls

[{'name': 'wikipedia',
  'args': {'query': 'FCT Abuja weather'},
  'id': '0kpqpc506',
  'type': 'tool_call'},
 {'name': 'tavily_search_results_json',
  'args': {'query': 'FCT Abuja weather today'},
  'id': 'x9prpb5hc',
  'type': 'tool_call'}]

Create Chatbot with LangGraph


In [ ]:
# Create State Schema
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
# Define Node
def tool_calling_llm(state:State):
    return {
        'messages': [llm_with_tools.invoke(state['messages'])]
    }
    
#Build graph
builder = StateGraph(State)
builder.add_node('tool_calling_llm', tool_calling_llm)
builder.add_node('tools', ToolNode(tools))

builder.add_edge(START, 'tool_calling_llm')
builder.add_conditional_edges(
    'tool_calling_llm', 
    tools_condition,
)

builder.add_edge('tools', END)

builder_graph = builder.compile()

#Display
display(Image(builder_graph.get_graph().draw_mermaid_png()))

ValueError: Found edge starting at unknown node 'tool_caling_llm'